In [1]:
import sys
import os
import shutil
import numpy as np
import time
from multiprocessing import Pool

# ==========================================
# 1. DYNAMIC PATH SETUP (No Hardcoded Paths)
# ==========================================
def setup_paths():
    """
    Automatically finds the project root containing 'src' 
    and adds it to the Python path.
    """
    current_path = os.path.abspath(os.getcwd())
    
    root_found = False
    for _ in range(5): 
        if os.path.exists(os.path.join(current_path, "src")):
            if current_path not in sys.path:
                sys.path.append(current_path)
            print(f"✅ Project root detected at: {current_path}")
            root_found = True
            return current_path
        current_path = os.path.dirname(current_path)
    
    if not root_found:
        print("⚠️ Warning: Could not auto-detect 'src' folder. Imports may fail.")
        return os.getcwd()

PROJECT_ROOT = setup_paths()

# ==========================================
# 2. IMPORTS (After Path Fix)
# ==========================================
try:
    import openmc
    from src.cross_section_read import CrossSectionReader
    from src.vt_calc import VelocitySampler
    from src.simulation import simulate_single_particle
    from src.material import Material
    from src.medium import Region, Sphere
    from src.tally import Tally
    from src.random_number_generator import RNGHandler
    from src.settings import Settings
except ImportError as e:
    print(f"\n❌ CRITICAL ERROR: {e}")
    print("Please ensure you are running this from within the PyNeut project structure.")
    sys.exit(1)

# ==========================================
# 3. CONFIGURATION
# ==========================================
BASE_DATA_PATH = os.path.join(PROJECT_ROOT, "endfb")
BENCHMARK_DIR = os.path.join(PROJECT_ROOT, "benchmark_runs")
N_PARTICLES = 10000 

if os.path.exists(BASE_DATA_PATH):
    print("Loading Cross Section Reader...")
    READER = CrossSectionReader(BASE_DATA_PATH)
else:
    print(f"❌ Error: Data folder not found at {BASE_DATA_PATH}")
    sys.exit(1)

# ==========================================
# 4. OPENMC RUNNER
# ==========================================
def run_openmc(energy_ev, radius=10.0, material_name="Pb208"):
    print(f"   [OpenMC] Simulating {energy_ev/1e6:.1f} MeV on {material_name}...")
    
    run_dir = os.path.join(BENCHMARK_DIR, f"openmc_{int(energy_ev)}")
    if os.path.exists(run_dir): shutil.rmtree(run_dir)
    os.makedirs(run_dir)
    
    # Material
    mat = openmc.Material(name='Target')
    mat.add_nuclide(material_name, 1.0)
    mat.set_density('g/cm3', 11.35)
    materials = openmc.Materials([mat])
    
    # Geometry
    surf = openmc.Sphere(r=radius, boundary_type='vacuum')
    cell = openmc.Cell(region=-surf, fill=mat)
    geometry = openmc.Geometry([cell])
    
    # Settings
    settings = openmc.Settings()
    settings.run_mode = 'fixed source'
    settings.batches = 20
    settings.particles = N_PARTICLES // 20
    settings.output = {'summary': False}
    
    source = openmc.Source()
    source.space = openmc.stats.Point((0, 0, 0))
    source.angle = openmc.stats.Isotropic()
    source.energy = openmc.stats.Discrete([energy_ev], [1.0])
    settings.source = source
    
    # Tallies
    tallies = openmc.Tallies()
    
    t_leak = openmc.Tally(name='leakage')
    t_leak.filters = [openmc.SurfaceFilter(surf)]
    t_leak.scores = ['current']
    tallies.append(t_leak)
    
    e_bins = np.linspace(0, energy_ev, 101)
    t_spec = openmc.Tally(name='spectrum')
    t_spec.filters = [openmc.SurfaceFilter(surf), openmc.EnergyFilter(e_bins)]
    t_spec.scores = ['current']
    tallies.append(t_spec)

    # Run
    cwd = os.getcwd()
    try:
        os.chdir(run_dir)
        model = openmc.Model(geometry=geometry, materials=materials, settings=settings, tallies=tallies)
        sp_path = model.run(output=False) 
        
        with openmc.StatePoint(sp_path) as sp:
            leak_val = sp.get_tally(name='leakage').mean[0][0][0]
            
            spec = sp.get_tally(name='spectrum')
            df = spec.get_pandas_dataframe()
            df['E_mid'] = (df['energy low [eV]'] + df['energy high [eV]']) / 2
            total_w = df['mean'].sum()
            avg_e = (df['E_mid'] * df['mean']).sum() / total_w if total_w > 0 else 0.0
            
    finally:
        os.chdir(cwd)
        
    return leak_val, avg_e

# ==========================================
# 5. PYNEUT RUNNER (UPDATED FOR BANKING)
# ==========================================
def run_pyneut(energy_ev, radius=10.0, material="Pb208"):
    print(f"   [PyNeut] Simulating {energy_ev/1e6:.1f} MeV on {material}...")
    
    # 1. Material
    lead = Material(name="Lead", density=11.35, atomic_mass=207.97, atomic_weight_ratio=207.2)
    N = lead.number_density
    A = lead.atomic_weight_ratio
    mass_kg = lead.kg_mass
    sampler = VelocitySampler(mass=mass_kg)
    
    # 2. Geometry
    mediums = [
        Region(surfaces=[Sphere((0,0,0), radius)], name="Sphere", priority=1, element=material)
    ]
    
    # 3. Settings
    settings = Settings(mode="shielding", particles=N_PARTICLES)
    
    # 4. Particles
    rngs = [RNGHandler(seed=12345 + i) for i in range(N_PARTICLES)]
    particle_states = [
        {
            "x": 0.0, "y": 0.0, "z": 0.0,
            "theta": np.arccos(1 - 2 * rng.random()), 
            "phi": rng.uniform(0, 2 * np.pi),
            "has_interacted": False,
            "energy": energy_ev, 
            "weight": 1.0
        } for rng in rngs
    ]
    
    READER.get_inelastic_components(material, energy_ev, N)

    args = [
        (state, READER, mediums, A, N, sampler, None, False, rng, settings)
        for state, rng in zip(particle_states, rngs)
    ]
    
    with Pool() as pool:
        results = pool.map(simulate_single_particle, args)
        
    # 6. Analyze (FIXED LOGIC HERE)
    total_weight_escaped = 0.0
    escaped_energies = []
    
    for res in results:
        # NEW: Check if *any* weight escaped (banking returns result="simulated")
        if res["final_weight"] > 0:
            w = res["final_weight"]
            e = res["final_energy"]
            total_weight_escaped += w
            escaped_energies.append((e, w))
            
    leakage = total_weight_escaped / N_PARTICLES
    
    if escaped_energies:
        num = sum(e * w for e, w in escaped_energies)
        den = sum(w for _, w in escaped_energies)
        avg_e = num / den
    else:
        avg_e = 0.0
        
    return leakage, avg_e

# ==========================================
# 6. MAIN BENCHMARK LOOP
# ==========================================
def print_comparison(title, omc_res, py_res):
    leak_diff = abs(omc_res[0] - py_res[0]) / omc_res[0] * 100 if omc_res[0] > 0 else 0.0
    en_diff = abs(omc_res[1] - py_res[1]) / omc_res[1] * 100 if omc_res[1] > 0 else 0.0
    
    print("\n" + "="*65)
    print(f"BENCHMARK RESULT: {title}")
    print("="*65)
    print(f"{'Metric':<20} | {'OpenMC':<15} | {'PyNeut':<15} | {'Diff %':<10}")
    print("-" * 65)
    print(f"{'Leakage Fraction':<20} | {omc_res[0]:<15.5f} | {py_res[0]:<15.5f} | {leak_diff:<10.2f}")
    print(f"{'Avg Escape Energy':<20} | {omc_res[1]:<15.2f} | {py_res[1]:<15.2f} | {en_diff:<10.2f}")
    print("="*65 + "\n")

if __name__ == "__main__":
    print(f"\n🚀 STARTING VALIDATION SUITE (N={N_PARTICLES})")
    print(f"Data Path: {BASE_DATA_PATH}\n")

    # --- CASE 1: ELASTIC REGIME (2.0 MeV) ---
    omc_1 = run_openmc(2.0e6)
    py_1 = run_pyneut(2.0e6)
    print_comparison("Elastic Scattering (2.0 MeV)", omc_1, py_1)
    
    # --- CASE 2: INELASTIC REGIME (14.0 MeV) ---
    omc_2 = run_openmc(14.0e6)
    py_2 = run_pyneut(14.0e6)
    print_comparison("Inelastic Scattering (14.0 MeV)", omc_2, py_2)

    # --- CASE 3: PURE INELASTIC (5.0 MeV) ---
    print("\n--- Running TEST 3: Pure Inelastic (5.0 MeV) ---")
    omc_3 = run_openmc(5.0e6)
    py_3 = run_pyneut(5.0e6)
    print_comparison("Pure Inelastic (5.0 MeV)", omc_3, py_3)

    # --- CASE 4: THE SWEET SPOT (4.0 MeV) ---
    print("\n--- Running TEST 4: Discrete Levels Only (4.0 MeV) ---")
    omc_4 = run_openmc(4.0e6)
    py_4 = run_pyneut(4.0e6)
    print_comparison("Discrete Inelastic (4.0 MeV)", omc_4, py_4)
    
    print("✅ Benchmark Suite Complete.")

✅ Project root detected at: /home/jovyan/work/endfb/PyNeut
Loading Cross Section Reader...

🚀 STARTING VALIDATION SUITE (N=10000)
Data Path: /home/jovyan/work/endfb/PyNeut/endfb

   [OpenMC] Simulating 2.0 MeV on Pb208...


/openmc_venv/lib/python3.11/site-packages/openmc/source.py:656: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn("This class is deprecated in favor of 'IndependentSource'", FutureWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=1.
  warn(msg, IDWarning)


   [PyNeut] Simulating 2.0 MeV on Pb208...

BENCHMARK RESULT: Elastic Scattering (2.0 MeV)
Metric               | OpenMC          | PyNeut          | Diff %    
-----------------------------------------------------------------
Leakage Fraction     | 0.99980         | 0.99968         | 0.01      
Avg Escape Energy    | 1959259.85      | 1962321.23      | 0.16      

   [OpenMC] Simulating 14.0 MeV on Pb208...


/openmc_venv/lib/python3.11/site-packages/openmc/source.py:656: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn("This class is deprecated in favor of 'IndependentSource'", FutureWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=4.
  warn(msg, IDWarning)


   [PyNeut] Simulating 14.0 MeV on Pb208...

BENCHMARK RESULT: Inelastic Scattering (14.0 MeV)
Metric               | OpenMC          | PyNeut          | Diff %    
-----------------------------------------------------------------
Leakage Fraction     | 1.50600         | 1.50214         | 0.26      
Avg Escape Energy    | 5257567.07      | 5273335.93      | 0.30      


--- Running TEST 3: Pure Inelastic (5.0 MeV) ---
   [OpenMC] Simulating 5.0 MeV on Pb208...


/openmc_venv/lib/python3.11/site-packages/openmc/source.py:656: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn("This class is deprecated in favor of 'IndependentSource'", FutureWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=7.
  warn(msg, IDWarning)


   [PyNeut] Simulating 5.0 MeV on Pb208...

BENCHMARK RESULT: Pure Inelastic (5.0 MeV)
Metric               | OpenMC          | PyNeut          | Diff %    
-----------------------------------------------------------------
Leakage Fraction     | 0.99980         | 0.99966         | 0.01      
Avg Escape Energy    | 2770744.15      | 2775414.87      | 0.17      


--- Running TEST 4: Discrete Levels Only (4.0 MeV) ---
   [OpenMC] Simulating 4.0 MeV on Pb208...


/openmc_venv/lib/python3.11/site-packages/openmc/source.py:656: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn("This class is deprecated in favor of 'IndependentSource'", FutureWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=10.
  warn(msg, IDWarning)


   [PyNeut] Simulating 4.0 MeV on Pb208...

BENCHMARK RESULT: Discrete Inelastic (4.0 MeV)
Metric               | OpenMC          | PyNeut          | Diff %    
-----------------------------------------------------------------
Leakage Fraction     | 0.99970         | 0.99976         | 0.01      
Avg Escape Energy    | 2756985.10      | 2769805.49      | 0.47      

✅ Benchmark Suite Complete.


In [3]:
import sys
import os
import shutil
import numpy as np
import warnings
from multiprocessing import Pool

# Suppress OpenMC warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP PATHS & IMPORTS
# ==========================================
def setup_paths():
    """Recursively find the project root containing 'src'."""
    current_path = os.path.abspath(os.getcwd())
    for _ in range(5): 
        if os.path.exists(os.path.join(current_path, "src")):
            if current_path not in sys.path: sys.path.append(current_path)
            return current_path
        current_path = os.path.dirname(current_path)
    return os.getcwd()

PROJECT_ROOT = setup_paths()
sys.path.append(PROJECT_ROOT)

try:
    import openmc
    from src.cross_section_read import CrossSectionReader
    from src.vt_calc import VelocitySampler
    from src.simulation import simulate_single_particle
    from src.material import Material
    from src.medium import Region, Sphere, Cylinder, Box, Plane
    from src.random_number_generator import RNGHandler
    from src.settings import Settings
except ImportError as e:
    print(f"❌ IMPORTS FAILED: {e}")
    print("Ensure you are running this from the project root or src parent.")
    sys.exit(1)

BASE_DATA_PATH = os.path.join(PROJECT_ROOT, "endfb")
BENCHMARK_DIR = os.path.join(PROJECT_ROOT, "benchmark_final")
N_PARTICLES = 10000 

if os.path.exists(BASE_DATA_PATH):
    print(f"Loading Reader from: {BASE_DATA_PATH}")
    READER = CrossSectionReader(BASE_DATA_PATH)
else:
    print(f"❌ Data missing at {BASE_DATA_PATH}"); sys.exit(1)

# ==========================================
# 2. TEST CASES
# ==========================================
# Note: 'dims' for Cyl is [Radius, +/- Z_Bound]
TEST_CASES = [
    # Baseline: Heavy Nucleus Elastic
    {"name": "Pb208 Elastic", "geo": "sphere", "dims": [10.0], "E": 2e6, "el": "Pb208", "rho": 11.35, "A": 207.2},
    
    # Baseline: Multiplier / Inelastic
    {"name": "Pb208 (n,2n)",  "geo": "sphere", "dims": [10.0], "E": 14e6, "el": "Pb208", "rho": 11.35, "A": 207.2},
    
    # Shielding: Iron Slab
    {"name": "Fe56 Slab",     "geo": "slab",   "dims": [5.0],  "E": 14e6, "el": "Fe56",  "rho": 7.87,  "A": 55.4},
    
    # Shielding: Light Nucleus Slab
    {"name": "Al27 Thin",     "geo": "slab",   "dims": [2.0],  "E": 1e6,  "el": "Al27",  "rho": 2.70,  "A": 26.98},
    
    # Moderator: Graphite (Finite Cylinder)
    # Using 'C12' based on your file listing. Standard ENDF often uses 'C0' (Natural Carbon).
    {"name": "Graphite Cyl",  "geo": "cyl",     "dims": [10.0, 20.0], "E": 2e6, "el": "C12", "rho": 2.26, "A": 12.01},
    
    # Multiplier: Beryllium
    {"name": "Be9 (n,2n)",    "geo": "sphere", "dims": [10.0], "E": 14e6, "el": "Be9", "rho": 1.85, "A": 9.01},
]

# ==========================================
# 3. OPENMC RUNNER
# ==========================================
def run_openmc(case):
    print(f"   [OpenMC] {case['name']}...")
    run_dir = os.path.join(BENCHMARK_DIR, f"omc_{case['name'].replace(' ', '_').replace('(','').replace(')','')}")
    if os.path.exists(run_dir): shutil.rmtree(run_dir)
    os.makedirs(run_dir)
    
    # --- Material Definition ---
    mat = openmc.Material(name='Target')
    try:
        # Handle Nuclide (Pb208) vs Element (C0/Natural)
        # Using simple heuristic: if it ends in digits, try nuclide, else element
        nuc_name = case['el']
        # Special map for Graphite if using C0 in OpenMC vs C12 in PyNeut
        if nuc_name == "C12": nuc_name = "C0" 

        try:
            mat.add_nuclide(nuc_name, 1.0)
        except:
            # Fallback for naturals like C0, Fe0 if exact iso missing
            mat.add_element(case['el'].rstrip('0123456789'), 1.0)
    except Exception as e:
        print(f"OpenMC Mat Error: {e}")
        return 0.0, 0.0

    mat.set_density('g/cm3', case['rho'])
    materials = openmc.Materials([mat])
    
    # --- Geometry Definition ---
    surfs = []
    if case['geo'] == "sphere":
        surf = openmc.Sphere(r=case['dims'][0], boundary_type='vacuum')
        cell = openmc.Cell(region=-surf, fill=mat)
        surfs = [surf]
        
    elif case['geo'] == "slab":
        d = case['dims'][0]
        # Slab bounded in X, infinite in Y/Z (vacuum boundaries applied for tallying)
        min_x = openmc.XPlane(x0=-d, boundary_type='vacuum')
        max_x = openmc.XPlane(x0=+d, boundary_type='vacuum')
        yr = openmc.YPlane(y0=-50, boundary_type='vacuum')
        yl = openmc.YPlane(y0=50, boundary_type='vacuum')
        zr = openmc.ZPlane(z0=-50, boundary_type='vacuum')
        zl = openmc.ZPlane(z0=50, boundary_type='vacuum')
        cell = openmc.Cell(region=+min_x & -max_x & +yr & -yl & +zr & -zl, fill=mat)
        surfs = [min_x, max_x]
        
    elif case['geo'] == "cyl":
        R, H = case['dims']
        cyl = openmc.ZCylinder(r=R, boundary_type='vacuum')
        bot = openmc.ZPlane(z0=-H, boundary_type='vacuum')
        top = openmc.ZPlane(z0=+H, boundary_type='vacuum')
        cell = openmc.Cell(region=-cyl & +bot & -top, fill=mat)
        surfs = [cyl, bot, top]

    geometry = openmc.Geometry([cell])
    
    # --- Settings ---
    settings = openmc.Settings()
    settings.run_mode = 'fixed source'
    settings.batches = 20
    settings.particles = N_PARTICLES // 20
    settings.output = {'summary': False}
    # Point source at origin, Isotropic
    settings.source = openmc.Source(
        space=openmc.stats.Point((0,0,0)), 
        angle=openmc.stats.Isotropic(), 
        energy=openmc.stats.Discrete([case['E']], [1.0])
    )
    
    # --- Tallies ---
    tallies = openmc.Tallies()
    
    # 1. Leakage (Current through surface)
    t_leak = openmc.Tally(name='leakage')
    t_leak.filters = [openmc.SurfaceFilter(surfs)]
    t_leak.scores = ['current']
    tallies.append(t_leak)
    
    # 2. Energy Spectrum (Current vs Energy)
    e_bins = np.linspace(0, case['E']*1.01, 51)
    t_spec = openmc.Tally(name='spectrum')
    t_spec.filters = [openmc.SurfaceFilter(surfs), openmc.EnergyFilter(e_bins)]
    t_spec.scores = ['current']
    tallies.append(t_spec)

    cwd = os.getcwd()
    try:
        os.chdir(run_dir)
        model = openmc.Model(geometry, materials, settings, tallies)
        sp_path = model.run(output=False)
        
        with openmc.StatePoint(sp_path) as sp:
            # Get Leakage (Sum of current over all surfaces)
            leak_val = np.sum(np.abs(sp.get_tally(name='leakage').mean.flatten()))
            
            # Get Avg Escape Energy
            spec = sp.get_tally(name='spectrum')
            df = spec.get_pandas_dataframe()
            df['E_mid'] = (df['energy low [eV]'] + df['energy high [eV]']) / 2
            total_w = df['mean'].abs().sum()
            avg_e = (df['E_mid'] * df['mean'].abs()).sum() / total_w if total_w > 0 else 0.0
            
    except Exception as e:
        print(f"OpenMC Run Failed: {e}")
        return 0.0, 0.0
    finally:
        os.chdir(cwd)
        
    return leak_val, avg_e

# ==========================================
# 4. PYNEUT RUNNER
# ==========================================
def run_pyneut(case):
    print(f"   [PyNeut] {case['name']}...")
    
    # Handle File Mapping (PyNeut expects exact H5 filename)
    el_file = case['el']
    
    # Construct Material
    # Atomic weight ratio passed as both atomic_mass and ratio for simplicity here
    # (Assuming single isotope/element material)
    mat_obj = Material(case['el'], case['rho'], case['A'], case['A']) 
    N = mat_obj.number_density
    
    # --- Geometry Construction ---
    if case['geo'] == "sphere":
        mediums = [Region(
            surfaces=[Sphere((0,0,0), case['dims'][0])], 
            operation="intersection", name="Sphere", priority=1, element=el_file
        )]
        
    elif case['geo'] == "slab":
        d = case['dims'][0]
        # Box from -d to d in X, large in Y/Z
        mediums = [Region(
            surfaces=[Box(-d, d, -50, 50, -50, 50)], 
            operation="intersection", name="Slab", priority=1, element=el_file
        )]
        
    elif case['geo'] == "cyl":
        R, H = case['dims']
        # FIXED: Finite Cylinder with Caps
        # Plane(A,B,C, D) -> Ax + By + Cz + D <= 0 (Inside)
        # Top Cap (z <= H):   z - H <= 0  -> Plane(0, 0, 1, H)
        # Bottom Cap (z >= -H): -z - H <= 0 -> Plane(0, 0, -1, H)
        mediums = [Region(
            surfaces=[
                Cylinder("z", R, (0,0,0)),
                Plane(0, 0, 1, H),   # Top Z=+H
                Plane(0, 0, -1, H)   # Bottom Z=-H
            ], 
            operation="intersection", name="Cyl", priority=1, element=el_file
        )]

    settings = Settings("shielding", N_PARTICLES)
    rngs = [RNGHandler(12345 + i) for i in range(N_PARTICLES)]
    
    # Initialize Source (Isotropic Point Source at Origin)
    states = [{
        "x": 0.0, "y": 0.0, "z": 0.0, 
        "theta": np.arccos(1 - 2 * r.random()), 
        "phi": r.uniform(0, 2 * np.pi), 
        "has_interacted": False, 
        "energy": case['E'], 
        "weight": 1.0
    } for r in rngs]

    # Pre-cache inelastic to avoid race conditions in multiprocessing
    try: READER.get_inelastic_components(el_file, case['E'], N)
    except: pass

    # Run Simulation
    # Note: Passing VelocitySampler with mass 1.0 (placeholder) as high E ignores thermal motion anyway
    args = [(s, READER, mediums, case['A'], N, VelocitySampler(1.0), None, False, r, settings) for s, r in zip(states, rngs)]
    
    with Pool() as pool: 
        results = pool.map(simulate_single_particle, args)
        
    # Analyze Results
    escaped_weight = 0.0
    escaped_energies = []
    
    for r in results:
        # Check result status or weight
        # If particle escaped, it contributes to leakage
        if r["final_weight"] > 0:
            escaped_weight += r["final_weight"]
            escaped_energies.append((r["final_energy"], r["final_weight"]))
            
    avg_e = sum(e*w for e,w in escaped_energies) / sum(w for _,w in escaped_energies) if escaped_energies else 0.0
    leakage_fraction = escaped_weight / N_PARTICLES
    
    return leakage_fraction, avg_e

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f"\n🚀 STARTING VALIDATION FINAL (N={N_PARTICLES})\n")
    print(f"Path: {PROJECT_ROOT}")
    
    for case in TEST_CASES:
        # Verify File Exists Before Running
        fname = case['el']
        if not os.path.exists(os.path.join(BASE_DATA_PATH, f"neutron/{fname}.h5")):
            print(f"⏩ Skipping {case['name']} (File {fname}.h5 not found)")
            continue
            
        omc_res = run_openmc(case)
        py_res = run_pyneut(case)
        
        print(f"\nRESULTS: {case['name']}")
        print(f"{'Metric':<10} | {'OpenMC':<10} | {'PyNeut':<10} | {'Diff %':<6}")
        print("-" * 45)
        
        # Calc Diffs
        l_diff = abs(omc_res[0] - py_res[0]) / omc_res[0] * 100 if omc_res[0] > 0 else 0
        e_diff = abs(omc_res[1] - py_res[1]) / omc_res[1] * 100 if omc_res[1] > 0 else 0
        
        print(f"{'Leakage':<10} | {omc_res[0]:<10.4f} | {py_res[0]:<10.4f} | {l_diff:<6.2f}")
        print(f"{'Energy':<10} | {omc_res[1]:<10.0f} | {py_res[1]:<10.0f} | {e_diff:<6.2f}")

    print("\n✅ Final Suite Complete.")

Loading Reader from: /home/jovyan/work/endfb/PyNeut/endfb

🚀 STARTING VALIDATION FINAL (N=10000)

Path: /home/jovyan/work/endfb/PyNeut
   [OpenMC] Pb208 Elastic...
   [PyNeut] Pb208 Elastic...

RESULTS: Pb208 Elastic
Metric     | OpenMC     | PyNeut     | Diff %
---------------------------------------------
Leakage    | 0.9998     | 0.9997     | 0.01  
Energy     | 1962031    | 1962138    | 0.01  
   [OpenMC] Pb208 (n,2n)...
   [PyNeut] Pb208 (n,2n)...

RESULTS: Pb208 (n,2n)
Metric     | OpenMC     | PyNeut     | Diff %
---------------------------------------------
Leakage    | 1.5060     | 1.5035     | 0.16  
Energy     | 5275196    | 5256556    | 0.35  
   [OpenMC] Fe56 Slab...
   [PyNeut] Fe56 Slab...

RESULTS: Fe56 Slab
Metric     | OpenMC     | PyNeut     | Diff %
---------------------------------------------
Leakage    | 1.1003     | 1.1501     | 4.52  
Energy     | 4732820    | 4544759    | 3.97  
   [OpenMC] Al27 Thin...
   [PyNeut] Al27 Thin...

RESULTS: Al27 Thin
Metric     |